In [ ]:
import uproot
import pandas as pd
import os

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
colors = [
    "#0b80c3",  # ATLAS blue
    "#c34e0b",  # Complementary
    "#c30b80",  # Triadic 1
    "#80c30b",  # Triadic 2
    "#0b24c3",  # Analogous 1
    "#0bc3aa",  # Analogous 2
    "#c30b24",  # Split 1
    "#c3aa0b",  # Split 2
    "#09669c",  # Monochromatic 1
    "#0d9aea"   # Monochromatic 2
]


# Load benchmark results

In [ ]:
results_dir = "csv"

results_df = pd.DataFrame()

for file in os.listdir(results_dir):
    temp_df = pd.read_csv(os.path.join(results_dir, file))
    temp_df["resultsFile"] = file
    results_df = pd.concat([results_df, temp_df], ignore_index=True)
    
# Drop empty columns
results_df = results_df.dropna(axis=1, how='all')

# Add compressor name
results_df['compressorName'] = results_df['compressor'].str.split('{').str[0]

# Add input size in MB
results_df['inputSizeMB'] = results_df['inputSizeBytes'] / (1024 * 1024)

results_df

In [ ]:
results_df[['dataFile', 'inputSizeMB']].drop_duplicates().sort_values(by='inputSizeMB')

# Plot compression ratio vs mean relative error

In [ ]:
small_file = "DAOD_PHYSLITE.37019878._000001.pool.root.1"
# small_MB_file = "DAOD_PHYSLITE.37019878._000001.pool.root.1"
medium_file = "DAOD_PHYSLITE.37019878._000018.pool.root.1"
large_file = "DAOD_PHYSLITE.37019878._000009.pool.root.1"

small_file_filter = results_df['dataFile'] == small_file
medium_file_filter = results_df['dataFile'] == medium_file
large_file_filter = results_df['dataFile'] == large_file
all_file_filter = results_df['dataFile'].isin([small_file, medium_file, large_file])
algorithm_filter = (results_df['algorithm'] == 3) | (results_df['compressorName'] == 'TruncCompressor')
quantity_filter = results_df['dataName'].str.contains('.pt')
error_filter = (results_df['meanRelErrorPct'] < 10) & (results_df['meanRelErrorPct'] > 0)

plot_df = results_df[algorithm_filter & quantity_filter & all_file_filter & error_filter]

# Remap compressor names
compressor_mapping = {
    'TruncCompressor': 'Bit Truncation',
    'SZ3Compressor': 'SZ3'
}

plot_df['compressorName'] = plot_df['compressorName'].map(compressor_mapping)

plot_df

In [ ]:
sz3_df = plot_df[(plot_df['compressorName'] == 'SZ3') & (plot_df['dataFile'].str.contains('000009'))]
trunc_df = plot_df[(plot_df['compressorName'] == 'Bit Truncation') & (plot_df['dataFile'].str.contains('000009'))]

sz3_df[['compressionRatio', 'meanRelErrorPct', 'outputSizeBytes']].sort_values(by='meanRelErrorPct')

In [ ]:
trunc_df[['compressionRatio', 'meanRelErrorPct', 'outputSizeBytes']].sort_values(by='meanRelErrorPct')

In [ ]:
# fig = make_subplots(
#     rows=1, cols=1,
#     subplot_titles=["Compression Ratio vs Mean Relative Error"],
#     specs=[[{"type": "scatter"}]]
# )

fig = px.scatter(
    plot_df,
    x="meanRelErrorPct",
    y="compressionRatio",
    facet_row="inputSizeMB",
    color="compressorName",
    color_discrete_sequence=colors[:2][::-1],
    title="Compression Ratio vs Mean Relative Error (%) for Jet Pt",
    category_orders={"inputSizeMB": sorted(plot_df['inputSizeMB'].unique())},
)

# Annotations should have format "Input Size = X MB"
# Only keep 2 decimal places for input size
fig.for_each_annotation(
    lambda a: a.update(text=f"Input Size = {float(a.text.split('=')[-1]):.2f} MB" if "=" in a.text else a.text)  # Clean up facet labels
)  # Remove row facet label

fig.update_layout(
    xaxis_title="Mean Relative Error (%)",
    yaxis_title="Compression Ratio",
    legend_title="Compressor",
    height=800,
    width=800,
    # template="plotly_dark",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# Make markers more prominent
fig.update_traces(
    marker=dict(
        size=10, 
        # opacity=0.8 if plot_df['compressorName'].str.contains('TruncCompressor') else 0.5,
        line=dict(width=1, color='DarkSlateGrey'),
    )
)

fig.update_yaxes(
    title_text="Compression Ratio",
)

# Add average lossless compression ratio line
avg_lossless_ratio = 1.45
fig.add_hline(
    y=avg_lossless_ratio,
    line_dash="dash",
    line_color="black",
    annotation_text="Average Lossless Compression Ratio",
    annotation_position="top right"
)

# Use dark background template
# fig.update_layout(
#     # plot_bgcolor='rgba(0, 0, 0, 0)',
#     # paper_bgcolor='rgba(0, 0, 0, 0)',
#     font=dict(color='white'),
#     xaxis=dict(showgrid=True, gridcolor='grey'),
#     yaxis=dict(showgrid=True, gridcolor='grey')
# )

# fig.show()
# write as high res png
fig.write_image("compression_ratio_vs_mean_relative_error.png", scale=3)